# ARIMA Models

So far, we have used several ways to represent time-series data and build models that use those representations to produce forecasts. Decomposition separates trend-cycle and seasonality. Exponential smoothing updates estimates of level, slope, and seasonal effects as observations arrive.

Today we will add one more tool: a family of models called ARIMA. These models combine regression on past values, differencing, and information in past forecast errors. After introducing the family, we will turn to the question of how to compare these forecasting methods and choose among them.

## References

- [Forecasting: Principles and Practice, the Pythonic Way, §§9.3–9.5](https://otexts.com/fpppy/09-arima.html#autoregressive-models): autoregression, moving-average models, the combined ARIMA equation, and special cases. The equation and special-case table below follow §9.5.
- [FPP Pythonic §9.7](https://otexts.com/fpppy/09-arima.html#arima-modelling-in-statsforecast): automatic model selection and the closing process diagram.
- [StatsForecast AutoARIMA](https://nixtlaverse.nixtla.io/statsforecast/src/core/models.html#autoarima): the automatic selection interface.
- [FPP Pythonic §9.9](https://otexts.com/fpppy/09-arima.html#seasonal-arima-models): seasonal terms and their combination with nonseasonal terms.
- [Allen B. Downey, Think Stats, Chapter 12](https://allendowney.github.io/ThinkStats/chap12.html#autoregression): an intuitive treatment using the nuclear-generation series from our earlier lectures.
- [statsmodels ARIMA](https://www.statsmodels.org/v0.14.6/generated/statsmodels.tsa.arima.model.ARIMA.html): the Python interface used in the examples.

## Autoregression

Linear regression predicts a response using a weighted combination of predictors. In an **autoregression**, the predictors are earlier values of that same response. An AR model of order $p$ uses the preceding $p$ observations:

$$
y_t=c+\phi_1y_{t-1}+\cdots+\phi_py_{t-p}+\varepsilon_t
$$

Here, $c$ is a constant and the $\phi$ coefficients are fitted weights. The weighted past values supply the prediction; $\varepsilon_t$ is the new, unpredictable error at time $t$.

This new surprise is called an **innovation**: the part of the observation that could not be predicted using earlier information. We model innovations as white noise: mean zero, constant variance, and no autocorrelation.

### A One-Step Prediction

Suppose an illustrative AR(1) model has constant 10 and coefficient 0.8. If the latest observation is 60, the next prediction is

$$
\hat y_{t+1|t}=c+\phi_1y_t=10+0.8(60)=58
$$

The notation $\hat y_{t+1|t}$ means a prediction for time $t+1$ using information through time $t$. We replace the unknown error term with its mean, zero.

If the next observed value is 61, the forecast error is $61-58=3$. The one-step prediction following that observed value would be $10+0.8(61)=58.8$.

To calculate the long-run mean, suppose the process has a constant mean $\mu$. Both $y_t$ and $y_{t-1}$ then have that same mean. Taking expectations of the AR(1) equation, with mean-zero errors, gives

$$
\mu=c+\phi_1\mu
$$

Rearrange to solve for the mean:

$$
\mu=\frac{c}{1-\phi_1}=\frac{10}{1-0.8}=50
$$

A stationary AR(1) model requires $|\phi_1|<1$. Our coefficient of 0.8 satisfies that condition. Its mean is 50 even though individual observations fluctuate around that value.

### Fit an AR Model

The next cell constructs 240 monthly observations from that AR(1) rule, with random innovations of standard deviation 2. This controlled example lets us compare estimated parameters with the rule that generated the observations. The initial value is 50; the coefficients remain fixed throughout.

In [ ]:
from pathlib import Path
from urllib.request import urlopen

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from statsmodels.tsa.arima.model import ARIMA

plt.rcParams.update({"figure.figsize": (10, 4), "font.size": 12,
                     "axes.spines.top": False, "axes.spines.right": False})
# Fix the random seed so the constructed series is reproducible.
rng = np.random.default_rng(7130)
innovations = rng.normal(0, 2, 240)
values = np.empty(240)
previous = 50.0
# Feed each new observation into the next AR(1) update.
for t, innovation in enumerate(innovations):
    values[t] = 10 + 0.8 * previous + innovation
    previous = values[t]
ar_series = pd.Series(values, index=pd.date_range("2000-01-01", periods=240, freq="MS"),
                      name="Constructed measurement")

`normal(0, 2, 240)` draws 240 innovations with mean zero and standard deviation 2. The loop calculates an observation, stores it, and uses it as `previous` for the following update. `pd.date_range(..., freq="MS")` attaches month-start labels to the constructed observations.

Use `ARIMA` with `order=(1, 0, 0)` for one AR lag, no differencing, and no MA terms. The constructor specifies the model; `.fit()` estimates its parameters from the supplied history. We withhold the last twelve observations so that fitting cannot use them.

In [ ]:
# Reserve the final twelve observations for the forecast comparison.
ar_training = ar_series.iloc[:-12]
ar_test = ar_series.iloc[-12:]
ar_fit = ARIMA(ar_training, order=(1, 0, 0), trend="c").fit()
display(ar_fit.params.to_frame("Estimated parameter").round(3))

`.iloc[:-12]` takes the history before the final twelve observations; `.iloc[-12:]` takes those twelve outcomes. `trend="c"` includes a fitted constant term. The fitted result stores named parameter estimates in `.params`.

The fitted mean is about 49.76 and the AR coefficient is about 0.773, compared with generating values of 50 and 0.8. Sampling variation means estimates need not recover the generating values exactly.

For this undifferenced model, statsmodels reports `const` as the fitted mean, rather than the recurrence constant $c$ in our equation. The equivalent recurrence constant is $c=\mu(1-\phi_1)$. `ar.L1` is the lag-one coefficient; `sigma2` is the estimated innovation variance.

Generate twelve future predictions from the fitted history, then plot the withheld observations alongside these fixed-origin forecasts. Compare the path with the generating mean of 50.

In [ ]:
# Forecast all twelve months from the same final training observation.
ar_forecast = ar_fit.forecast(steps=len(ar_test))

fig, ax = plt.subplots()
ar_training.iloc[-36:].plot(ax=ax, label="Available history")
ar_test.plot(ax=ax, label="Withheld observations", marker="o")
ar_forecast.plot(ax=ax, label="AR(1) forecast", linestyle="--")
ax.axhline(50, color="gray", linewidth=1, label="Generating mean")
ax.set(title="AR(1): Forecasts Return Toward the Fitted Mean",
       xlabel="Month", ylabel="Constructed measurement")
ax.legend()
plt.tight_layout()
plt.show()
plt.close(fig)

`.forecast(steps=len(ar_test))` returns twelve predictions labeled with the months following the training history. The fitted parameters stay fixed throughout that forecast path. The plot includes the final 36 training observations to show where the forecast begins.

At a fixed origin, future innovations are unknown. With a fitted coefficient between zero and one, the forecast approaches the fitted mean as the horizon grows. The observations continue to fluctuate because they contain new innovations.

## Moving-Average Models

Earlier observations are one source of predictive information. Recent innovations can also affect later observations. We estimate past innovations using the differences between observations and the model's one-step predictions, so past forecast errors supply information for the next prediction.

An MA($q$) model combines current and past innovations:

$$
y_t=c+\varepsilon_t+\theta_1\varepsilon_{t-1}+\cdots+\theta_q\varepsilon_{t-q}
$$

Each observation is a constant plus a weighted combination of innovations. The weights need not be positive or sum to one. We do not directly observe the innovations; fitting the model estimates them along with the coefficients. The estimated innovations supply the past forecast errors used in prediction.

This **moving-average model** differs from **moving-average smoothing**. Our earlier rolling averages smoothed observed values to estimate a trend-cycle. Here, earlier innovations help predict a future observation.

### A Recent Surprise Changes the Prediction

Consider the illustrative MA(1) model

$$
y_t=50+\varepsilon_t+0.6\varepsilon_{t-1}
$$

If the latest estimated innovation is 3, the next point forecast is $50+0.6(3)=51.8$. The unknown next innovation has expected value zero. At horizon two, this MA(1) model forecasts 50 because both innovations in that future observation are still unknown.

The next cell shows how one innovation of 3 affects this model when all other innovations are zero. This isolates the effect of one shock; it is not a typical random realization.

In [ ]:
shock = pd.Series([0.0, 3.0, 0.0, 0.0], index=[0, 1, 2, 3], name="Innovation")
# Align the preceding innovation with each current time.
ma_effect = 50 + shock + 0.6 * shock.shift(1, fill_value=0)
display(pd.DataFrame({"Innovation": shock, "Observation": ma_effect})
        .rename_axis("Time"))

`shock.shift(1, fill_value=0)` moves each innovation to the following row, supplying the previous innovation for the MA(1) equation. The zero at the first row represents the assumed innovation before this example begins. The table places each innovation beside the observation it helps generate.

The innovation raises the observation at time 1 by 3 and at time 2 by 1.8. Its direct effect ends after that. This finite persistence differs from the repeated propagation of an AR(1) effect through later observations.

## Combine AR, Differencing, and MA

A trend can make the relationship among original levels unstable. We have already used differencing to work with changes instead. Write $y'_t$ for the series after the chosen number of ordinary differences. For one difference,

$$
y'_t=y_t-y_{t-1}
$$

We choose differences when needed to obtain an approximately stationary series, retrieving the stable-mean and dependence-by-lag assumptions discussed earlier.

AR and MA contributions then add together to model that transformed series. Following FPP equation 9.1,

$$
y'_t=c+\phi_1y'_{t-1}+\cdots+\phi_py'_{t-p}+\theta_1\varepsilon_{t-1}+\cdots+\theta_q\varepsilon_{t-q}+\varepsilon_t
$$

Differencing changes the response we model. After predicting a change, we recover the predicted level by adding that change to the preceding level.

### Read the Model Orders

**ARIMA** stands for **AutoRegressive Integrated Moving Average**. “Integrated” means the model describes a series whose differences are stationary. Forecast differences are accumulated to recover forecasts on the original scale. We write a nonseasonal model as ARIMA($p,d,q$):

| Setting | Meaning |
| --- | --- |
| $p$ | Number of consecutive AR lags |
| $d$ | Number of ordinary differencing operations |
| $q$ | Number of consecutive MA lags |

The orders specify the structure. Fitting estimates the coefficients and innovation variance within that structure. More terms supply more flexibility; whether they improve future predictions is an evaluation question.

### Familiar Special Cases

FPP Table 9.1 connects several familiar models to the same family:

| Model | ARIMA specification |
| --- | --- |
| White noise | ARIMA(0,0,0), without a constant |
| Random walk | ARIMA(0,1,0), without a constant |
| Random walk with drift | ARIMA(0,1,0), with a constant in the differenced equation |
| Autoregression | ARIMA($p$,0,0) |
| Moving-average model | ARIMA(0,0,$q$) |

In statsmodels, a random walk uses `trend="n"`; adding drift uses `trend="t"`, a linear trend in levels that becomes a constant after differencing.

## Seasonal ARIMA

Seasonal ARIMA extends the same ideas to observations and innovations separated by a seasonal period. We write

$$
\operatorname{ARIMA}(p,d,q)(P,D,Q)_m
$$

| Setting | Meaning for monthly observations with yearly seasonality |
| --- | --- |
| $m=12$ | Twelve observations per seasonal cycle |
| $P$ | AR terms at lags 12, 24, and so on |
| $D$ | Number of seasonal differences, each comparing values twelve months apart |
| $Q$ | MA terms at lags 12, 24, and so on |

The lowercase orders still describe adjacent-period behavior. For example, one ordinary difference compares this month with last month; one seasonal difference compares this month with the same month last year.

### Fit a Seasonal Model to Nuclear Generation

Return to monthly US nuclear generation, measured in GWh, from January 2001 through June 2024. We use the unchanged EIA extract from 12a and keep its July 2019–June 2024 holdout. A forecasting procedure must learn from the history available before that holdout.

The next cell loads the existing course extract and restores its monthly index. It uses the published 11a copy when the local file is unavailable, including in Colab.

In [ ]:
# Read the existing course extract locally, with the published copy as a fallback.
candidates = [
    Path("../12a-basic-forecasting/data/us-nuclear-generation.csv"),
    Path("content/lectures/12a-basic-forecasting/data/us-nuclear-generation.csv"),
    Path("lectures/12a-basic-forecasting/data/us-nuclear-generation.csv"),
]
local_data = next((path for path in candidates if path.exists()), None)
if local_data is not None:
    frame = pd.read_csv(local_data)
else:
    url = ("https://raw.githubusercontent.com/olearydj/INSY7130/main/"
           "lectures/11a-time-series-and-decomposition/data/us-nuclear-generation.csv")
    with urlopen(url, timeout=30) as response:
        frame = pd.read_csv(response)
frame["month"] = pd.to_datetime(frame["month"], format="%b %Y")
nuclear = frame.set_index("month")["generation_gwh"].asfreq("MS")
assert nuclear.index.is_unique and nuclear.notna().all()
# Split chronologically before fitting either forecasting procedure.
training = nuclear.iloc[:-60]
test = nuclear.iloc[-60:]
display(pd.DataFrame({"First month": [training.index[0], test.index[0]],
                      "Last month": [training.index[-1], test.index[-1]],
                      "Observations": [len(training), len(test)]},
                     index=["Training", "Test"]))

`pd.to_datetime` reads the month labels, and `.asfreq("MS")` establishes a month-start series. The assertion checks unique dates and complete values. The positional slices retain 222 training months through June 2019 and reserve the following 60 months for comparison, as shown in the table.

Use ARIMA(1,0,0)(0,1,0)$_{12}$ as an illustrative starting candidate: take year-over-year differences, then model adjacent-month dependence in those changes with AR(1). This is a deliberately small specification to make the settings readable.

Supply the original series to `ARIMA`; it applies the requested differences internally. `trend="n"` excludes a deterministic trend. This seasonal model replaces the separate decomposition step used in 12a.

In [ ]:
# Supply original levels; the model applies the requested seasonal difference.
seasonal_model = ARIMA(training, order=(1, 0, 0),
                       seasonal_order=(0, 1, 0, 12), trend="n")
seasonal_fit = seasonal_model.fit()
seasonal_forecast = seasonal_fit.forecast(steps=len(test))
parameter_table = seasonal_fit.params.to_frame("Estimated parameter").round(3)
# Match each description to its parameter name.
parameter_table["Description"] = pd.Series({
    "ar.L1": "Lag-one AR coefficient for year-over-year changes",
    "sigma2": "Innovation variance (GWh²)",
})
display(parameter_table)

`.fit()` estimates the model coefficients from `training`; `.forecast()` extends the fitted model for the requested number of months. The returned forecasts are on the original scale. `.params.to_frame()` turns the named estimates into a table, and pandas aligns the description entries by parameter name.

The fitted lag-one coefficient is about 0.185. The seasonal difference is a change in generation from the same month a year earlier. The AR coefficient describes dependence between successive year-over-year changes.

Compare the forecast with the withheld observations. The model was fitted once; it receives no new observations while forecasting these sixty months.

In [ ]:
fig, ax = plt.subplots()
training.iloc[-36:].plot(ax=ax, label="Available history")
test.plot(ax=ax, label="Withheld observations", alpha=0.75)
seasonal_forecast.plot(ax=ax, label="Seasonal ARIMA forecast", linestyle="--")
ax.set(title="Nuclear Generation: One Origin, Sixty Forecast Months",
       xlabel="Year", ylabel="Generation (GWh)")
ax.legend()
plt.tight_layout()
plt.show()
plt.close(fig)

The forecast repeats a seasonal pattern, but several later troughs are lower than it predicts. The plot supplies a reason to measure errors and compare alternatives.

Where does this procedure miss the level or seasonal pattern? Would the same ranking hold if we made a fresh forecast every month, or only needed the next three months?

## Automatic ARIMA Selection

We specified the orders in the previous example. Could software choose them instead? **AutoARIMA** searches within the ARIMA family, fitting candidate models and selecting one using a criterion that balances fit against model complexity. StatsForecast uses corrected Akaike information criterion, AICc, by default; smaller values are preferred among the candidates it compares.

AutoARIMA also chooses ordinary and seasonal differencing orders when we leave them unspecified. We still supply the seasonal period: twelve observations per year. The result is an already fitted ARIMA model, ready to forecast. We do not need a separate fitting call to statsmodels afterward.

### Let Software Select the Orders

We use StatsForecast's `AutoARIMA`, a different interface from the statsmodels `ARIMA` used above. The course environment includes `statsforecast`; in Colab, run `%pip install statsforecast==2.0.1` once before running this section. The next cell uses only the same 222 training observations. All six orders are left to the automatic procedure; `.fit()` selects the structure and estimates its parameters. Its default search has limits and uses a stepwise procedure, so automatic selection does not mean trying every possible model.

In [ ]:
from statsforecast.arima import arima_string
from statsforecast.models import AutoARIMA

# Let the procedure choose all orders using the training observations.
auto_fit = AutoARIMA(season_length=12).fit(training.to_numpy())
print("Selected model:", arima_string(auto_fit.model_).strip())
display(pd.Series(auto_fit.model_["coef"], name="Estimated coefficient")
        .to_frame().round(3))

`.to_numpy()` supplies the numeric training observations in chronological order. AutoARIMA stores its selected fitted model in `.model_`. `arima_string` formats the selected orders, and `["coef"]` retrieves the estimated coefficients for the table.

Here, the procedure selects ARIMA(1,0,0)(1,1,1) at period 12, with drift. It retains one seasonal difference and adds seasonal AR and MA terms. In the table, `ar1` is the ordinary lag-one coefficient, `sar1` and `sma1` are the seasonal AR and MA coefficients, and `drift` is a fitted linear change in level per month. These choices and estimates use only the training history.

### Forecast the Same Withheld Months

Forecast the same sixty withheld months with the automatically selected model. We will compare both procedures on identical dates, using the observations held out from selection and fitting.

In [ ]:
auto_prediction = auto_fit.predict(h=len(test))
# Restore the monthly labels for comparison with the same holdout.
auto_forecast = pd.Series(auto_prediction["mean"], index=test.index,
                          name="AutoARIMA forecast")
assert auto_forecast.notna().all()

`auto_prediction["mean"]` extracts the point forecasts from the returned dictionary. `pd.Series(..., index=test.index)` pairs them with the sixty target months. The assertion checks that every target has a forecast.

Overlay the automatically selected forecast and our specified model. Look for differences in their predicted level, seasonal shape, and longer-run behavior.

In [ ]:
fig, ax = plt.subplots()
training.iloc[-36:].plot(ax=ax, label="Available history", color="gray")
test.plot(ax=ax, label="Withheld observations", color="black", alpha=0.7)
seasonal_forecast.plot(ax=ax, label="Specified ARIMA", linestyle="--")
auto_forecast.plot(ax=ax, label="AutoARIMA", linestyle=":", linewidth=2)
ax.set(title="Nuclear Generation: Specified and Automatically Selected ARIMA",
       xlabel="Year", ylabel="Generation (GWh)")
ax.legend()
plt.tight_layout()
plt.show()
plt.close(fig)

Automatic selection answers which candidate the training-based procedure prefers. Does that candidate make better forecasts for our purpose? We will carry both forecast series into evaluation alongside the simpler baselines.

## Using ARIMA in Practice

We have introduced how ARIMA models work and how to fit one. Choosing and checking a useful model involves more than producing a forecast:

- **Differencing:** Choose ordinary and seasonal differences that make the modeled series approximately stationary. More differencing is not automatically better.
- **Model orders:** Choose which past observations and errors to include. More terms increase flexibility and the number of parameters to estimate.
- **Residuals:** Check whether the fitted model leaves temporal dependence unexplained. Residuals resembling white noise are a useful diagnostic, not proof that forecasts will be accurate.
- **Forecast performance:** Compare against simple baselines and other methods at the horizons that matter.

### The ARIMA Modeling Process

FPP Figure 9.12 summarizes the process. Both manual and automatic selection begin with understanding the data and return to checking the residuals. The diagram's automatic branch labels the function `ARIMA()`; in our Python implementation, that branch corresponds to StatsForecast's `AutoARIMA`, while statsmodels' `ARIMA` fits the orders we supply.

Read the two selection paths and where they rejoin.

![FPP Figure 9.12: General process for forecasting using an ARIMA model](https://otexts.com/fpppy/figs/arimaflowchart.png)

Source: Hyndman and colleagues, [*Forecasting: Principles and Practice, the Pythonic Way*, Figure 9.12](https://otexts.com/fpppy/09-arima.html#modelling-procedure), reproduced unchanged. The figure includes Box–Cox transformations, unit-root tests, and PACF analysis as parts of the broader workflow; we have not developed those techniques here.

Software can help select candidate models, but its selection criterion does not establish that a model will forecast well for our purpose. That requires evaluation using observations unavailable during selection and fitting. The diagram ends with calculating forecasts; our next step is to evaluate them.

## Compare Forecasting Procedures

We now have several plausible ways to forecast: simple baselines, regression, decomposition-based forecasts, exponential smoothing, and ARIMA. A plausible representation and a good historical fit do not establish which will work best on future observations.

Our next task is to make the comparison match the forecasting decision:

- What horizon matters?
- Which observations would be available at each forecast origin?
- What simple baseline must a more complex model improve on?
- Which errors matter most, and how consistently does one method outperform another?

We will use these questions to extend our earlier chronological holdout comparisons into a fuller evaluation procedure.

---

Auburn University / Industrial and Systems Engineering<br>
INSY 7130, Pattern Discovery and Time Series Analysis<br>
© Copyright Danny J. O'Leary.

Conceptual sequence, the combined nonseasonal equation, and the special-case table adapted from Hyndman and colleagues, *Forecasting: Principles and Practice, the Pythonic Way*, Chapter 9. Numerical teaching examples and implementation are course additions. The nuclear-generation measurements are the unchanged EIA extract accompanying Downey's *Think Stats*; see the [12a data provenance](../12a-basic-forecasting/data/README.md).